# 🏦 JPMC Consumer Banking - Hands-on Lab
## Notebook 03b: Data Transformation with Stored Procedures + Tasks (Imperative Approach)

---

### What You'll Learn
- Implementing the **same transformation logic** as Notebook 03 using stored procedures
- Building a **Task DAG** with dependencies to orchestrate execution
- **Comparing** declarative (Dynamic Tables) vs imperative (SP + Tasks) approaches
- Monitoring task execution history

### Why Compare?

Dynamic Tables are ideal for straightforward transformations where the logic is a single SELECT statement. But stored procedures + tasks give you:
- **Conditional logic** (IF/ELSE, loops)
- **Multi-step operations** (INSERT then UPDATE then DELETE)
- **Error handling** (TRY/CATCH with custom logging)
- **External calls** (APIs, notifications)
- **Complex business rules** that don't fit in a single query

### Task DAG Architecture (mirrors the DT DAG from Notebook 03)
```
TASK_ROOT (Hourly Schedule)
  ├── TASK_CUSTOMER_360 (AFTER TASK_ROOT)
  ├── TASK_TXN_SUMMARY (AFTER TASK_ROOT)
  └── TASK_EXEC_DASHBOARD (AFTER TASK_CUSTOMER_360, TASK_TXN_SUMMARY)
```

In [ ]:
-- Setup context
USE ROLE ACCOUNTADMIN;
USE DATABASE JPMC_CONSUMER_BANKING_HOL;
USE SCHEMA HOL_LAB;
USE WAREHOUSE HOL_MEDIUM_WH;

## Step 1: Create Target Tables for SP Output

Unlike Dynamic Tables (which auto-create their output table), stored procedures write into pre-existing tables.

In [ ]:
-- =============================================================================
-- TARGET TABLES (SP will write results here)
-- Using _SP suffix to differentiate from DT outputs
-- =============================================================================

CREATE OR REPLACE TABLE CUSTOMER_360_SP LIKE CUSTOMER_360;
CREATE OR REPLACE TABLE TXN_SUMMARY_SP LIKE TXN_SUMMARY;
CREATE OR REPLACE TABLE EXECUTIVE_DASHBOARD_SP LIKE EXECUTIVE_DASHBOARD;

-- Add audit columns to track SP execution
ALTER TABLE CUSTOMER_360_SP ADD COLUMN _REFRESHED_AT TIMESTAMP DEFAULT CURRENT_TIMESTAMP();
ALTER TABLE TXN_SUMMARY_SP ADD COLUMN _REFRESHED_AT TIMESTAMP DEFAULT CURRENT_TIMESTAMP();
ALTER TABLE EXECUTIVE_DASHBOARD_SP ADD COLUMN _REFRESHED_AT TIMESTAMP DEFAULT CURRENT_TIMESTAMP();

## Step 2: Create Stored Procedures

Each procedure implements the same logic as its corresponding Dynamic Table, but with added features:
- **Logging** - tracks execution time and row counts
- **Error handling** - catches and reports failures
- **Full refresh pattern** - TRUNCATE + INSERT (could be MERGE for incremental)

In [ ]:
-- =============================================================================
-- STORED PROCEDURE: SP_REFRESH_CUSTOMER_360
-- Same logic as DT CUSTOMER_360
-- =============================================================================

CREATE OR REPLACE PROCEDURE SP_REFRESH_CUSTOMER_360()
RETURNS VARCHAR
LANGUAGE SQL
EXECUTE AS CALLER
AS
BEGIN
    LET start_time TIMESTAMP := CURRENT_TIMESTAMP();
    LET row_count INTEGER := 0;
    
    BEGIN
        -- Full refresh: truncate and reload
        TRUNCATE TABLE CUSTOMER_360_SP;
        
        INSERT INTO CUSTOMER_360_SP (
            CUSTOMER_ID, FIRST_NAME, LAST_NAME, EMAIL, SEGMENT, RISK_RATING,
            CUSTOMER_SINCE, IS_ACTIVE, TOTAL_ACCOUNTS, ACTIVE_ACCOUNTS,
            TOTAL_BALANCE, AVG_ACCOUNT_BALANCE, LAST_ACTIVITY,
            CHECKING_BALANCE, SAVINGS_BALANCE, MONEY_MARKET_BALANCE,
            CITY, STATE, _REFRESHED_AT
        )
        SELECT
            c.CUSTOMER_ID, c.FIRST_NAME, c.LAST_NAME, c.EMAIL, c.SEGMENT, c.RISK_RATING,
            c.CREATED_AT, c.IS_ACTIVE,
            COUNT(DISTINCT a.ACCOUNT_ID),
            SUM(CASE WHEN a.STATUS = 'ACTIVE' THEN 1 ELSE 0 END),
            SUM(a.BALANCE),
            AVG(a.BALANCE),
            MAX(a.LAST_ACTIVITY_DATE),
            SUM(CASE WHEN a.ACCOUNT_TYPE = 'CHECKING' THEN a.BALANCE ELSE 0 END),
            SUM(CASE WHEN a.ACCOUNT_TYPE = 'SAVINGS' THEN a.BALANCE ELSE 0 END),
            SUM(CASE WHEN a.ACCOUNT_TYPE = 'MONEY_MARKET' THEN a.BALANCE ELSE 0 END),
            c.ADDRESS_JSON:city::VARCHAR,
            c.ADDRESS_JSON:state::VARCHAR,
            CURRENT_TIMESTAMP()
        FROM CUSTOMERS c
        LEFT JOIN ACCOUNTS a ON c.CUSTOMER_ID = a.CUSTOMER_ID
        GROUP BY ALL;
        
        row_count := SQLROWCOUNT;
        
        RETURN 'SUCCESS: ' || :row_count || ' rows refreshed in ' || 
               DATEDIFF('second', :start_time, CURRENT_TIMESTAMP()) || ' seconds';
    EXCEPTION
        WHEN OTHER THEN
            RETURN 'ERROR: ' || SQLERRM;
    END;
END;

In [ ]:
-- =============================================================================
-- STORED PROCEDURE: SP_REFRESH_TXN_SUMMARY
-- =============================================================================

CREATE OR REPLACE PROCEDURE SP_REFRESH_TXN_SUMMARY()
RETURNS VARCHAR
LANGUAGE SQL
EXECUTE AS CALLER
AS
BEGIN
    LET start_time TIMESTAMP := CURRENT_TIMESTAMP();
    LET row_count INTEGER := 0;
    
    BEGIN
        TRUNCATE TABLE TXN_SUMMARY_SP;
        
        INSERT INTO TXN_SUMMARY_SP (
            ACCOUNT_ID, TXN_DAY, TXN_MONTH, CATEGORY, CHANNEL, TXN_TYPE,
            TXN_COUNT, TOTAL_AMOUNT, AVG_AMOUNT, MAX_AMOUNT, MIN_AMOUNT,
            COMPLETED_COUNT, FAILED_COUNT, REVERSED_COUNT, HIGH_VALUE_TXN_COUNT,
            _REFRESHED_AT
        )
        SELECT
            ACCOUNT_ID,
            DATE_TRUNC('DAY', TO_TIMESTAMP(TXN_DATE)),
            DATE_TRUNC('MONTH', TO_TIMESTAMP(TXN_DATE)),
            CATEGORY, CHANNEL, TXN_TYPE,
            COUNT(*),
            SUM(AMOUNT), AVG(AMOUNT), MAX(AMOUNT), MIN(AMOUNT),
            SUM(CASE WHEN STATUS = 'COMPLETED' THEN 1 ELSE 0 END),
            SUM(CASE WHEN STATUS = 'FAILED' THEN 1 ELSE 0 END),
            SUM(CASE WHEN STATUS = 'REVERSED' THEN 1 ELSE 0 END),
            SUM(CASE WHEN AMOUNT > 1000 THEN 1 ELSE 0 END),
            CURRENT_TIMESTAMP()
        FROM TRANSACTIONS
        GROUP BY ALL;
        
        row_count := SQLROWCOUNT;
        RETURN 'SUCCESS: ' || :row_count || ' rows in ' || 
               DATEDIFF('second', :start_time, CURRENT_TIMESTAMP()) || 's';
    EXCEPTION
        WHEN OTHER THEN
            RETURN 'ERROR: ' || SQLERRM;
    END;
END;

In [ ]:
-- =============================================================================
-- STORED PROCEDURE: SP_REFRESH_EXECUTIVE_DASHBOARD (Python-based)
-- (Depends on CUSTOMER_360_SP and TXN_SUMMARY_SP being refreshed first)
-- =============================================================================

CREATE OR REPLACE PROCEDURE SP_REFRESH_EXECUTIVE_DASHBOARD()
RETURNS VARCHAR
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python')
HANDLER = 'run'
EXECUTE AS CALLER
AS
$$
import time
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import *

def run(session):
    start_time = time.time()
    
    try:
        # Truncate target table
        session.sql("TRUNCATE TABLE EXECUTIVE_DASHBOARD_SP").collect()
        
        # Load source tables
        c360 = session.table("CUSTOMER_360_SP")
        accounts = session.table("ACCOUNTS")
        txn_summary = session.table("TXN_SUMMARY_SP")
        
        # Join Customer 360 with Accounts and Transaction Summary
        joined_df = c360.join(accounts, c360["CUSTOMER_ID"] == accounts["CUSTOMER_ID"], "left") \
            .join(txn_summary, accounts["ACCOUNT_ID"] == txn_summary["ACCOUNT_ID"], "left") \
            .filter(F.col("TXN_MONTH") == F.date_trunc("MONTH", F.current_date()))
        
        # Aggregate to build executive dashboard
        dashboard_df = joined_df.group_by(
            c360["STATE"], c360["SEGMENT"]
        ).agg(
            F.count_distinct(c360["CUSTOMER_ID"]).alias("TOTAL_CUSTOMERS"),
            F.sum(F.when(c360["IS_ACTIVE"] == True, F.lit(1)).otherwise(F.lit(0))).alias("ACTIVE_CUSTOMERS"),
            F.avg(c360["TOTAL_BALANCE"]).alias("AVG_CUSTOMER_BALANCE"),
            F.sum(c360["TOTAL_BALANCE"]).alias("TOTAL_DEPOSITS"),
            F.sum(F.col("TXN_COUNT")).alias("MONTHLY_TXN_COUNT"),
            F.sum(F.col("TOTAL_AMOUNT")).alias("MONTHLY_TXN_VOLUME"),
            F.avg(F.col("AVG_AMOUNT")).alias("AVG_TXN_SIZE"),
            F.sum(F.col("FAILED_COUNT")).alias("MONTHLY_FAILED_TXNS"),
            F.round(
                F.sum(F.col("FAILED_COUNT")) / F.coalesce(F.sum(F.col("TXN_COUNT")), F.lit(1)) * F.lit(100), 
                F.lit(3)
            ).alias("FAILURE_RATE_PCT"),
            F.sum(F.col("HIGH_VALUE_TXN_COUNT")).alias("HIGH_VALUE_TXNS")
        ).with_column("_REFRESHED_AT", F.current_timestamp())
        
        # Write results
        dashboard_df.write.mode("append").save_as_table("EXECUTIVE_DASHBOARD_SP")
        
        row_count = session.table("EXECUTIVE_DASHBOARD_SP").count()
        duration = round(time.time() - start_time, 1)
        
        return f"SUCCESS: {row_count} rows in {duration}s (Python SP)"
        
    except Exception as e:
        return f"ERROR: {str(e)}"
$$;

## Step 3: Test the Stored Procedures

In [ ]:
-- Test each procedure manually
CALL SP_REFRESH_CUSTOMER_360();
CALL SP_REFRESH_TXN_SUMMARY();
CALL SP_REFRESH_EXECUTIVE_DASHBOARD();

## Step 4: Create Task DAG (Orchestration)

Now we wire the stored procedures into a Task DAG that mirrors the DT dependency structure. The root task runs on a schedule, and child tasks execute after their parents complete.

In [ ]:
-- =============================================================================
-- TASK DAG - Orchestrating SPs with dependency management
-- =============================================================================

-- Root task: runs every hour (triggers the entire pipeline)
CREATE OR REPLACE TASK TASK_ROOT
    WAREHOUSE = HOL_MEDIUM_WH
    SCHEDULE = 'USING CRON 0 * * * * America/New_York'
    COMMENT = 'Root task - triggers hourly refresh pipeline'
AS
    SELECT 1;  -- No-op, just triggers children

-- Level 1: Independent tasks (run in parallel after root)
CREATE OR REPLACE TASK TASK_CUSTOMER_360
    WAREHOUSE = HOL_MEDIUM_WH
    AFTER TASK_ROOT
    COMMENT = 'Refresh Customer 360 view'
AS
    CALL SP_REFRESH_CUSTOMER_360();

CREATE OR REPLACE TASK TASK_TXN_SUMMARY
    WAREHOUSE = HOL_MEDIUM_WH
    AFTER TASK_ROOT
    COMMENT = 'Refresh Transaction Summary'
AS
    CALL SP_REFRESH_TXN_SUMMARY();

-- Level 2: Dependent task (runs after its parents complete)
CREATE OR REPLACE TASK TASK_EXEC_DASHBOARD
    WAREHOUSE = HOL_MEDIUM_WH
    AFTER TASK_CUSTOMER_360, TASK_TXN_SUMMARY
    COMMENT = 'Refresh Executive Dashboard (after Customer360 + TxnSummary)'
AS
    CALL SP_REFRESH_EXECUTIVE_DASHBOARD();

In [ ]:
-- =============================================================================
-- ENABLE THE TASK DAG (resume from root - cascades to children)
-- =============================================================================

-- Resume all tasks (bottom-up order required)
ALTER TASK TASK_EXEC_DASHBOARD RESUME;
ALTER TASK TASK_CUSTOMER_360 RESUME;
ALTER TASK TASK_TXN_SUMMARY RESUME;
ALTER TASK TASK_ROOT RESUME;

-- Verify task DAG
SHOW TASKS IN SCHEMA HOL_LAB;

In [ ]:
-- Manually execute the root task to test the full DAG
EXECUTE TASK TASK_ROOT;

## Step 5: Monitor Task Execution

In [ ]:
-- =============================================================================
-- TASK MONITORING
-- =============================================================================

-- View task execution history
SELECT NAME, STATE, SCHEDULED_TIME, COMPLETED_TIME, RETURN_VALUE,
    DATEDIFF('second', SCHEDULED_TIME, COMPLETED_TIME) AS DURATION_SECONDS
FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(
    SCHEDULED_TIME_RANGE_START => DATEADD(HOUR, -2, CURRENT_TIMESTAMP()),
    RESULT_LIMIT => 20
))
ORDER BY SCHEDULED_TIME DESC;

-- View task dependencies
SELECT * FROM TABLE(INFORMATION_SCHEMA.TASK_DEPENDENTS(
    TASK_NAME => 'TASK_ROOT',
    RECURSIVE => TRUE
));

## Step 6: Side-by-Side Comparison

| Aspect | Dynamic Tables (NB 03) | SP + Tasks (NB 03b) |
|--------|----------------------|---------------------|
| **Lines of code** | ~200 (just SELECT statements) | ~400+ (DDL + SP bodies + Task DDL) |
| **Orchestration** | Automatic - Snowflake manages DAG | Manual - define AFTER dependencies |
| **Incremental refresh** | Built-in (automatic change tracking) | Must code MERGE logic yourself |
| **Error handling** | Automatic retry | Custom TRY/CATCH blocks |
| **Monitoring** | DYNAMIC_TABLE_REFRESH_HISTORY | TASK_HISTORY |
| **Conditional logic** | Not supported (single SELECT) | Full IF/ELSE, loops, multi-step |
| **Cost model** | Pay for refresh compute only when data changes | Pay every scheduled run regardless |
| **Schema management** | Auto-managed by DT definition | Manual CREATE TABLE + ALTER |
| **Best for** | Standard transformations, aggregations | Complex business logic, multi-step ETL |

### Recommendation
- **Use Dynamic Tables** when your transformation is a single SELECT (90% of cases)
- **Use SP + Tasks** when you need conditional logic, multi-step operations, or external API calls
- **Mix both** - use DTs for the data pipeline, SPs for business operations (statements, alerts)

In [ ]:
-- =============================================================================
-- CLEANUP: Suspend tasks after demo (to avoid ongoing charges)
-- =============================================================================

ALTER TASK TASK_ROOT SUSPEND;
-- Child tasks automatically suspend when root is suspended

-- Verify
SELECT NAME, STATE FROM TABLE(INFORMATION_SCHEMA.TASK_DEPENDENTS(
    TASK_NAME => 'TASK_ROOT', RECURSIVE => TRUE
));